# Chapter 13: Supervised Learning — Regression Algorithms
## Notebook Pendamping Praktik Interaktif

Notebook ini memandu Anda mengeksplorasi, memodelkan, meregularisasi, dan mengevaluasi algoritma regresi:
1. **Eksplorasi Data Telemetri Konsumsi Energi Industri**.
2. **Simple & Multiple Linear Regression (Ordinary Least Squares)**.
3. **Polynomial Regression & Bias–Variance Trade-off**.
4. **Regularisasi L2 (Ridge) vs L1 (Lasso) vs ElasticNet**.
5. **Evaluasi Metrik (MAE, MSE, RMSE, R² & Fenomena R² Negatif)**.
6. **Analisis Diagnostik Residual (Homoskedastisitas & Normalitas)**.
7. **Benchmark Komparasi Model & Penetapan Model Produksi**.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Membaca dataset telemetri konsumsi daya
data_path = Path("../code/data/equipment_power_demand.csv")
df = pd.read_csv(data_path)
print(f"Dimensi Dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
print(f"Target Kontinu: power_consumption_kw (Rentang: {df['power_consumption_kw'].min():.1f} - {df['power_consumption_kw'].max():.1f} kW)")
df.head(3)


Dimensi Dataset: 300 baris x 7 kolom
Target Kontinu: power_consumption_kw (Rentang: 172.7 - 466.8 kW)


### 1. Pembagian Dataset & Multiple Linear Regression (OLS)
Memisahkan 80% data latih dan 20% data uji, lalu melatih model regresi linier multivariat.


In [2]:
feature_cols = [
    "ambient_temp_c", "operating_load_pct", "machine_speed_rpm",
    "inlet_pressure_bar", "vibration_severity_mms", "equipment_age_months"
]
X = df[feature_cols]
y = df["power_consumption_kw"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

mlr = LinearRegression()
mlr.fit(X_train, y_train)

y_pred = mlr.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Intercept (b)    : {mlr.intercept_:.4f} kW")
print(f"Test RMSE        : {rmse:.2f} kW")
print(f"Test R^2 Score   : {r2:.4f}")
print("Koefisien Fitur:")
for col, c in zip(feature_cols, mlr.coef_):
    print(f"  - {col:<24}: {c:>8.4f} kW / satuan")


Intercept (b)    : 3.3049 kW
Test RMSE        : 5.37 kW
Test R^2 Score   : 0.9951
Koefisien Fitur:
  - ambient_temp_c          :   0.6347 kW / satuan
  - operating_load_pct      :   2.4640 kW / satuan
  - machine_speed_rpm       :   0.0702 kW / satuan
  - inlet_pressure_bar      :   2.5984 kW / satuan
  - vibration_severity_mms  :   5.2861 kW / satuan
  - equipment_age_months    :   0.1510 kW / satuan


### 2. Polynomial Regression (Menangkap Kelengkungan Termal)
Menggunakan `PolynomialFeatures` untuk memodelkan hubungan non-linear tanpa mengubah linearitas parameter.


In [3]:
poly_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("linear", LinearRegression())
])
poly_pipe.fit(X_train, y_train)

poly_pred = poly_pipe.predict(X_test)
poly_rmse = np.sqrt(mean_squared_error(y_test, poly_pred))
poly_r2 = r2_score(y_test, poly_pred)

print(f"Jumlah Fitur Setelah Ekspansi Kuadratik: {poly_pipe.named_steps['poly'].n_output_features_}")
print(f"Polynomial Deg 2 Test RMSE              : {poly_rmse:.2f} kW (Penurunan dari {rmse:.2f} kW)")
print(f"Polynomial Deg 2 Test R^2 Score         : {poly_r2:.4f}")


Jumlah Fitur Setelah Ekspansi Kuadratik: 27
Polynomial Deg 2 Test RMSE              : 4.12 kW (Penurunan dari 5.37 kW)
Polynomial Deg 2 Test R^2 Score         : 0.9971


### 3. Regularisasi L2 (Ridge) vs L1 (Lasso) vs ElasticNet
Mengevaluasi efek regularisasi terhadap pencegahan overfitting dan seleksi fitur.


In [4]:
ridge_pipe = Pipeline([("scaler", StandardScaler()), ("reg", Ridge(alpha=10.0, random_state=42))]).fit(X_train, y_train)
lasso_pipe = Pipeline([("scaler", StandardScaler()), ("reg", Lasso(alpha=0.5, random_state=42))]).fit(X_train, y_train)
elastic_pipe = Pipeline([("scaler", StandardScaler()), ("reg", ElasticNet(alpha=0.5, l1_ratio=0.5, random_state=42))]).fit(X_train, y_train)

print(f"Ridge Active Features : {np.sum(ridge_pipe.named_steps['reg'].coef_ != 0)}/{len(feature_cols)} (Norm L2 = {np.linalg.norm(ridge_pipe.named_steps['reg'].coef_):.2f})")
print(f"Lasso Active Features : {np.sum(lasso_pipe.named_steps['reg'].coef_ != 0)}/{len(feature_cols)} (Fitur dengan w=0 tereliminasi!)")
print(f"ElasticNet Features   : {np.sum(elastic_pipe.named_steps['reg'].coef_ != 0)}/{len(feature_cols)}")


Ridge Active Features : 6/6 (Norm L2 = 47.92)
Lasso Active Features : 5/6 (Fitur dengan w=0 tereliminasi!)
ElasticNet Features   : 6/6


### 4. Diagnostik Residual
Memeriksa apakah asumsi dasar OLS (nilai harapan galat nol dan varians konstan) terpenuhi.


In [5]:
residuals = y_test - poly_pred
print(f"Mean Residual (Asumsi 0) : {residuals.mean():.4f} kW")
print(f"Std Deviasi Residual     : {residuals.std():.4f} kW")
print(f"Min / Max Galat Residual : [{residuals.min():.2f}, {residuals.max():.2f}] kW")


Mean Residual (Asumsi 0) : -0.3644 kW
Std Deviasi Residual     : 4.1352 kW
Min / Max Galat Residual : [-9.82, 7.84] kW
